In [3]:

import os
import numpy as np
import torch
from PIL import Image
from sklearn.preprocessing import LabelEncoder
from facenet_pytorch import InceptionResnetV1
import os
import numpy as np
import torch
from PIL import Image
from sklearn.preprocessing import LabelEncoder
from facenet_pytorch import InceptionResnetV1

# 1. Load pre-trained FaceNet model for feature extraction
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
facenet = InceptionResnetV1(pretrained='vggface2').eval().to(device)

def extract_embedding(image_path):
    """Loads a cropped face image and returns a 512-d embedding vector."""
    img = Image.open(image_path).convert('RGB').resize((160, 160))
    # Convert PIL Image to PyTorch Tensor normalized between -1 and 1
    tensor = torch.tensor(np.array(img)).permute(2, 0, 1).float()
    tensor = (tensor - 127.5) / 128.0
    tensor = tensor.unsqueeze(0).to(device)
    
    with torch.no_grad():
        embedding = facenet(tensor).cpu().numpy().flatten()
    return embedding

# 2. Main Loop: Pair each embedding with its folder name (label)
dataset_dir = "dataset/cropped_faces"

X_list = []
y_string_list = []

# Loop over each player's folder (e.g., 'lionel_messi', 'neymar')
for player_name in sorted(os.listdir(dataset_dir)):
    player_folder = os.path.join(dataset_dir, player_name)
    
    # Skip non-directory files like .DS_Store
    if not os.path.isdir(player_folder):
        continue
        
    print(f"Processing images for: {player_name}")
    
    for image_name in os.listdir(player_folder):
        image_path = os.path.join(player_folder, image_name)
        
        try:
            # Step A: Extract the 512-d embedding vector
            embedding = extract_embedding(image_path)
            
            # Step B: APPEND BOTH AT THE SAME TIME
            X_list.append(embedding)
            y_string_list.append(player_name)
            
        except Exception as e:
            print(f"Skipping corrupt image {image_path}: {e}")

# 3. Convert lists into NumPy Arrays
X = np.array(X_list)                 # Shape: (Total_Images, 512)
raw_labels = np.array(y_string_list) # Shape: (Total_Images,)

# 4. Convert string player names into numeric labels (0, 1, 2, ...)
label_encoder = LabelEncoder()
y = label_encoder.fit_transform(raw_labels)

# Save the label encoder mapping for reference later
np.save("classes.npy", label_encoder.classes_)

print(f"\nSuccessfully processed dataset!")
print(f"X shape (embeddings): {X.shape}")
print(f"y shape (numeric labels): {y.shape}")


Processing images for: cristiano_ronaldo
Skipping corrupt image dataset/cropped_faces\cristiano_ronaldo\.pipeline_crops.json: cannot identify image file 'dataset/cropped_faces\\cristiano_ronaldo\\.pipeline_crops.json'
Processing images for: kylian_mbappe
Skipping corrupt image dataset/cropped_faces\kylian_mbappe\.pipeline_crops.json: cannot identify image file 'dataset/cropped_faces\\kylian_mbappe\\.pipeline_crops.json'
Processing images for: lamine_yamal
Skipping corrupt image dataset/cropped_faces\lamine_yamal\.pipeline_crops.json: cannot identify image file 'dataset/cropped_faces\\lamine_yamal\\.pipeline_crops.json'
Processing images for: lionel_messi
Skipping corrupt image dataset/cropped_faces\lionel_messi\.pipeline_crops.json: cannot identify image file 'dataset/cropped_faces\\lionel_messi\\.pipeline_crops.json'
Processing images for: neymar
Skipping corrupt image dataset/cropped_faces\neymar\.pipeline_crops.json: cannot identify image file 'dataset/cropped_faces\\neymar\\.pipeli

In [4]:

from sklearn.model_selection import train_test_split
from keras.models import Sequential
from keras.layers import Dense, Dropout
from keras.optimizers import Adam
from keras.callbacks import EarlyStopping
from keras.activations import relu, softmax
from keras.losses import SparseCategoricalCrossentropy

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

model = Sequential([
    Dense(256, activation=relu, input_shape=(512,)),
    Dropout(0.4),
    Dense(128, activation=relu),
    Dropout(0.3),
    Dense(len(label_encoder.classes_), activation=softmax)
])

model.compile(
    optimizer=Adam(learning_rate=0.001),
    loss=SparseCategoricalCrossentropy(),
    metrics=['accuracy'],
)
model.summary()

c:\Users\shuj8\miniconda3\envs\dl\Lib\site-packages\keras\src\layers\core\dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 256)            │       131,328 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 128)            │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 7)              │           903 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 165,127 (645.03 KB)

 Trainable params: 165,127 (645.03 KB)

 Non-trainable params: 0 (0.00 B)

In [5]:
early_stopping = EarlyStopping(
    monitor='val_loss',
    patience=15,
    restore_best_weights=True
)

history = model.fit(
    X_train, y_train,
    validation_data=(X_test, y_test),
    epochs=100,
    batch_size=16,
    callbacks=[early_stopping]
)

Epoch 1/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 1s 13ms/step - accuracy: 0.7656 - loss: 1.6442 - val_accuracy: 0.9877 - val_loss: 1.1281
Epoch 2/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.9844 - loss: 0.6580 - val_accuracy: 1.0000 - val_loss: 0.1888
Epoch 3/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.9969 - loss: 0.1161 - val_accuracy: 1.0000 - val_loss: 0.0407
Epoch 4/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.9969 - loss: 0.0372 - val_accuracy: 1.0000 - val_loss: 0.0245
Epoch 5/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 1.0000 - loss: 0.0208 - val_accuracy: 1.0000 - val_loss: 0.0200
Epoch 6/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 1.0000 - loss: 0.0097 - val_accuracy: 1.0000 - val_loss: 0.0169
Epoch 7/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 1.0000 - loss: 0.0080 - val_accuracy: 1.0000 - val_loss: 0.0136
Epoch 8/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 1.0000 - loss: 0.0061 - val_accuracy: 1.0000 - 

In [6]:
# Save the model
model.save("football_player_classifier.keras")

# Save class names mapping
np.save("classes.npy", label_encoder.classes_)

In [7]:
import numpy as np

# 1. Load saved embeddings and labels (or calculate X and y)
# If you didn't save X and y during training, re-run the extraction step to generate them
X = X_train  # Shape: (N, 512)
y = y_train  # Shape: (N,)
class_names = np.load("classes.npy")

centroids = {}

# 2. Compute normalized mean vector for each player class
for class_idx, class_name in enumerate(class_names):
    # Select embeddings belonging to this specific player
    player_embeddings = X[y == class_idx]
    
    # Calculate average vector (centroid)
    mean_vector = np.mean(player_embeddings, axis=0)
    
    # Normalize centroid to unit length (for cosine similarity)
    normalized_centroid = mean_vector / np.linalg.norm(mean_vector)
    centroids[class_idx] = normalized_centroid

# 3. Save centroids dictionary
np.save("player_centroids.npy", centroids)
print("Successfully computed and saved 'player_centroids.npy' for 7 players.")

Successfully computed and saved 'player_centroids.npy' for 7 players.
